# 10 A/B readout
Read the holdout test in data/holdout.csv against a decision rule written down before the analysis.

Step 1: set up the kit

In [1]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-10"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Kit folder found: atp-builder-kit


Step 2: set the threshold before you look

In [2]:
import json, math
from datetime import datetime, timezone

MIN_LIFT = 0.01  # the smallest gain worth shipping: 1 point of conversion (absolute)
RULE = ("SHIP if the lower end of the 95 percent interval is above 0 and the lift is at least MIN_LIFT. "
        "STOP if the upper end is below MIN_LIFT. Otherwise KEEP RUNNING. If the segment mix differs between arms, "
        "use the segment adjusted lift, and find out why the mix differs before acting.")
(OUTPUT / "ab_threshold.json").write_text(json.dumps(
    {"min_lift": MIN_LIFT, "rule": RULE, "set_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}, indent=1))
print("Saved before reading any data:", RULE)

Saved before reading any data: SHIP if the lower end of the 95 percent interval is above 0 and the lift is at least MIN_LIFT. STOP if the upper end is below MIN_LIFT. Otherwise KEEP RUNNING. If the segment mix differs between arms, use the segment adjusted lift, and find out why the mix differs before acting.


Step 3: load the test, n per arm

In [3]:
import pandas as pd

DATA = Path(os.environ.get("KIT_DATA_DIR", ROOT / "data"))
df = pd.read_csv(DATA / "holdout.csv")
arms = df.groupby("arm").agg(n=("user_id", "count"), conversions=("converted", "sum"), rate=("converted", "mean"))
arms

,n,conversions,rate
arm,,,
control,2000,299,0.1495
treatment,2000,219,0.1095


Step 4: the lift, with a 95 percent interval

In [4]:
Z = 1.96  # 95 percent, two sided

def diff_ci(c1, n1, c0, n0):
    # Difference of two rates, with the normal approximation. Fine for these sample sizes.
    p1, p0 = c1 / n1, c0 / n0
    se = math.sqrt(p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0)
    return p1 - p0, p1 - p0 - Z * se, p1 - p0 + Z * se

t, c = arms.loc["treatment"], arms.loc["control"]
pooled = diff_ci(t.conversions, t.n, c.conversions, c.n)
print(f"Pooled lift: {pooled[0]:+.4f} (95% interval {pooled[1]:+.4f} to {pooled[2]:+.4f}), "
      f"relative {pooled[0] / c.rate:+.1%}")

Pooled lift: -0.0400 (95% interval -0.0608 to -0.0192), relative -26.8%


Step 5: mix shift check across segments

In [5]:
MAX_SHARE_GAP = 0.05  # flag when a segment's share differs by more than 5 points between arms
share = pd.crosstab(df.arm, df.platform, normalize="index")
gap = (share.loc["treatment"] - share.loc["control"]).abs().max()
mix_shift = gap > MAX_SHARE_GAP
print("Share of each arm by platform:")
print(share.round(3))
print(f"Biggest share gap: {gap:.3f} -> mix shift: {mix_shift}\n")

seg_rows = []
for platform, part in df.groupby("platform"):
    a = part.groupby("arm").converted.agg(["sum", "count", "mean"])
    lift, low, high = diff_ci(a.loc["treatment", "sum"], a.loc["treatment", "count"],
                              a.loc["control", "sum"], a.loc["control", "count"])
    seg_rows.append({"platform": platform, "n_control": a.loc["control", "count"], "n_treatment": a.loc["treatment", "count"],
                     "rate_control": round(a.loc["control", "mean"], 4), "rate_treatment": round(a.loc["treatment", "mean"], 4),
                     "lift": round(lift, 4), "low": round(low, 4), "high": round(high, 4)})
segments = pd.DataFrame(seg_rows).set_index("platform")
segments

Share of each arm by platform:
platform   desktop  mobile
arm                       
control        0.7     0.3
treatment      0.3     0.7
Biggest share gap: 0.400 -> mix shift: True



,n_control,n_treatment,rate_control,rate_treatment,lift,low,high
platform,,,,,,,
desktop,1400,600,0.1950,0.2267,0.0317,-0.0077,0.0711
mobile,600,1400,0.0433,0.0593,0.0160,-0.0045,0.0364


Step 6: a segment adjusted lift

In [6]:
# Weight each segment's lift by its share of ALL users, so a lopsided mix cannot move the total.
weights = df.platform.value_counts(normalize=True)
lift = var = 0.0
for platform, part in df.groupby("platform"):
    a = part.groupby("arm").converted.agg(["mean", "count"])
    pt, pc = a.loc["treatment", "mean"], a.loc["control", "mean"]
    lift += weights[platform] * (pt - pc)
    var += weights[platform] ** 2 * (pt * (1 - pt) / a.loc["treatment", "count"] + pc * (1 - pc) / a.loc["control", "count"])
adjusted = (lift, lift - Z * math.sqrt(var), lift + Z * math.sqrt(var))
print(f"Pooled:   {pooled[0]:+.4f} ({pooled[1]:+.4f} to {pooled[2]:+.4f})")
print(f"Adjusted: {adjusted[0]:+.4f} ({adjusted[1]:+.4f} to {adjusted[2]:+.4f})")

Pooled:   -0.0400 (-0.0608 to -0.0192)
Adjusted: +0.0238 (+0.0016 to +0.0460)


Step 7: the decision, against the rule set in step 2

In [7]:
used, (lift, low, high) = ("segment adjusted", adjusted) if mix_shift else ("pooled", pooled)
if low > 0 and lift >= MIN_LIFT:
    decision = "SHIP"
elif high < MIN_LIFT:
    decision = "STOP"
else:
    decision = "KEEP RUNNING"
readout = (f"Decision: {decision}\nUsed the {used} lift {lift:+.4f} (95% interval {low:+.4f} to {high:+.4f}); "
           f"threshold MIN_LIFT {MIN_LIFT}.\nn per arm: control {int(c.n)}, treatment {int(t.n)}. Mix shift: {mix_shift}.\n"
           f"Pooled lift was {pooled[0]:+.4f}.")
if mix_shift:
    readout += ("\nThe arms got different mixes of users. In a randomized test that usually means assignment or "
                "logging is broken: find out why before you act on any lift, the adjusted one too.")
(OUTPUT / "ab_readout.md").write_text("# A/B readout\n\n" + readout.replace("\n", "\n\n") + "\n")
print(readout)

Decision: SHIP
Used the segment adjusted lift +0.0238 (95% interval +0.0016 to +0.0460); threshold MIN_LIFT 0.01.
n per arm: control 2000, treatment 2000. Mix shift: True.
Pooled lift was -0.0400.
The arms got different mixes of users. In a randomized test that usually means assignment or logging is broken: find out why before you act on any lift, the adjusted one too.
